# 2.3 Simpson's paradox

**Is this a plotting lesson or a statistics lesson? The statistics one.** Plotting is the
tool that makes the reversal visible and lets you dig into it — the point of this notebook is
a fact about aggregation, not a chart type.

The fact: an aggregate comparison and *every one* of its subgroup comparisons can point in
opposite directions, with nothing miscalculated anywhere. "Group A beats group B" is not yet
a complete statement until you've also asked "beats it within what?" — and that question has
no default answer, which is what makes this genuinely hard rather than a checkbox to tick.

In [ ]:
import pandas as pd
from goad_toolkit.visualizer import BarbellPlot, GroupedBarPlot, HighlightCategory, PlotSettings

from scripts.plots import BarPlot
from wa_analyzer.data import load_showcase

## 2.3.1 The setup

Berkeley graduate admissions, 1973 — one of the most-cited real examples there is, and it is
real: the university was sued over apparent bias against women applicants. The six largest
departments, the numbers as published. One row per department per gender: how many applied,
how many were admitted.

In [ ]:
berkeley = load_showcase("berkeley_admissions")
berkeley.head(6)

## 2.3.2 The wrong conclusion first

Pool every department together and compare admission rate by gender — the obvious first
chart, and the one a rushed analysis stops at.

In [ ]:
overall = berkeley.groupby("gender")[["applied", "admitted"]].sum().reset_index()
overall["rate"] = overall.admitted / overall.applied * 100

aggregate_settings = PlotSettings(
    figsize=(6, 4),
    title="Admitted, by gender, all departments pooled",
    xlabel="gender",
    ylabel="admission rate (%)",
    highlight=["men"],
)
bars = BarPlot(aggregate_settings)
bars.plot(data=overall, x="gender", y="rate", color="#cccccc")
bars.plot_on(HighlightCategory(aggregate_settings))

44.5% of men admitted, 30.4% of women. Fourteen points is not subtle, the sample is
thousands of applicants, and the obvious story writes itself.

**Before believing it, ask the one question that defends against this every time:** is there
a variable that differs between the groups being compared *and* also affects the outcome? If
one exists, the aggregate is not measuring what you think it is measuring until you have
compared within it. Here, the candidate is sitting right there in the data — `department`.

## 2.3.3 Diving into the subgroup

Same comparison, once for each department. `GroupedBarPlot` next to the pooled chart, so the
two are read side by side rather than one displacing the other from memory.

In [ ]:
berkeley["rate"] = berkeley.admitted / berkeley.applied * 100

split = PlotSettings(
    figsize=(13, 4),
    title="Berkeley 1973, two levels of aggregation",
    subplot_titles=["pooled: men admitted more often",
                    "per department: mostly the opposite"],
    xlabel="",
    ylabel="admission rate (%)",
)

host = BarPlot(split)
fig, axes = host.create_figure(n_plots=2)

host.plot_on_axes(BarPlot(split), axes[0], data=overall, x="gender", y="rate", color="#cccccc")
host.plot_on_axes(GroupedBarPlot(split), axes[1],
                  data=berkeley, x="department", y="rate", hue="gender")
fig.tight_layout()

The pooled comparison favours men. Most of the per-department bars do not. Nothing here is
miscalculated — both charts are correct statements about the same twelve rows.

The right panel above already shows the flip, department by department — but "which
departments, and by how much" is exactly what a table of numbers would answer, and this is a
visualisation course. A **barbell plot** draws the men/women pair for each department as two
dots joined by a line, so the gap itself — not just the two endpoints — is what you see.

In [ ]:
wide = berkeley.pivot(index="department", columns="gender", values="rate").reset_index()
wide["gap"] = wide.women - wide.men
wide = wide.sort_values("gap")

barbell_settings = PlotSettings(
    figsize=(9, 4),
    title="Admission rate by department: men vs women",
    xlabel="admission rate (%)",
    ylabel="department",
)
fig, ax = BarbellPlot(barbell_settings).plot(
    data=wide, category="department", start="men", end="women",
    start_label="men", end_label="women",
)

favouring_women = int((wide.gap > 0).sum())
print(f"women admitted at a higher rate in {favouring_women} of {len(wide)} departments")

Four departments have the crimson dot (women) to the right of the grey one (men); two do
not. That is the whole paradox in one picture: pooled, the grey side wins; department by
department, the crimson side mostly does.

## 2.3.4 Why it flips: the lurking variable

The mechanism is in the application counts, not the admission decisions. Plot the actual
selectivity of each department first — the thing the aggregate rate never showed you — then
plot who applied where, in the same order, so the two panels read as cause and effect.

In [ ]:
from matplotlib.patches import Patch

applied = berkeley.pivot(index="department", columns="gender", values="applied")
admitted = berkeley.pivot(index="department", columns="gender", values="admitted")

mechanism = pd.DataFrame({
    "department": applied.index,
    "overall_rate": (admitted.men + admitted.women) / (applied.men + applied.women) * 100,
    "women_share": applied.women / (applied.men + applied.women) * 100,
}).sort_values("overall_rate", ascending=False)

hardest_two = mechanism.nsmallest(2, "overall_rate").department.tolist()

mech_settings = PlotSettings(
    figsize=(13, 4),
    title="Selectivity predicts who applied where",
    subplot_titles=["overall admission rate, easiest to hardest department",
                    "% of applicants who are women, same order"],
    xlabel="department",
    highlight=hardest_two,
)

host = BarPlot(mech_settings)
fig, axes = host.create_figure(n_plots=2)

host.plot_on_axes(BarPlot(mech_settings), axes[0], data=mechanism, x="department",
                  y="overall_rate", order=mechanism.department, color="#cccccc")
host.plot_on_axes(HighlightCategory(mech_settings), axes[0])
axes[0].set_ylabel("admission rate (%)")

host.plot_on_axes(BarPlot(mech_settings), axes[1], data=mechanism, x="department",
                  y="women_share", order=mechanism.department, color="#cccccc")
host.plot_on_axes(HighlightCategory(mech_settings), axes[1])
axes[1].set_ylabel("% of applicants who are women")

axes[0].legend(
    handles=[
        Patch(facecolor=mech_settings.base_color, label="other departments"),
        Patch(facecolor=mech_settings.highlight_color, label="the two hardest to get into"),
    ],
    loc="upper right", fontsize=8,
)
fig.tight_layout()

Line the two panels up and the mechanism reads directly: the two most selective departments
(the legend, and the highlight, name them) sit on the right of the left panel and on the
right of the right panel too — the departments that admit almost nobody are exactly the ones
that drew a much larger share of women applicants. Men applied more to the departments on the
left, the ones that admit most people who try. The aggregate rate mostly measures *which
department people applied to*, not how any individual applicant was treated once there.

That is the resolution, and it is worth restating precisely: nobody in this dataset was
treated differently for being a woman *within a department*. The pooled number makes it look
that way only because department and gender are entangled in the applicant pool.

## 2.3.5 The template for your own data

The defence generalises past admissions data completely:

> **Every time an aggregate comparison between two groups looks decisive, ask: is there a
> variable that differs between the groups and also affects the outcome? If yes, compare
> within it — the same way, one subgroup at a time — before you believe the aggregate.**

The hard part is never the groupby. It is *noticing a candidate lurking variable exists*
before you have written the sentence claiming the aggregate result. A short list of the usual
suspects, for the kind of data this course works with:

- **Time period.** A platform, a channel or a group changes over months; pooling all of it
  together can hide (or manufacture) a trend that a per-period comparison would show plainly.
- **Tenure or activity level.** New members versus long-time regulars, casual posters versus
  the eight people who write half of everything — 02.4 is built entirely around this one.
- **Sub-channel or topic.** A comparison across two group chats, or two IRC channels, can be
  driven by what each one is *about* rather than by who is in it.
- **Device, platform, or export method.** Different timestamp formats, different message
  truncation rules — a technical difference that correlates with a group.
- **Weekday versus weekend, or time of day.** Any behaviour with its own daily or weekly
  rhythm can make two groups look different if one happens to be sampled more from one part
  of the cycle.

Checking one is always the same three steps: split the aggregate comparison by the candidate
variable (`groupby` two columns instead of one), plot each subgroup the same way you plotted
the aggregate — a `GroupedBarPlot`, or a barbell if there is a clear before/after — and look
for a reversal, not just a change in size. If the direction survives every subgroup, the
aggregate was fine. If it does not, you have found something more interesting than what you
started with.

---

**Where this goes next.** Simpson's paradox needs a hidden *grouping* variable to appear.
[02.4-same_trap](02.4-same_trap.ipynb) shows a version that needs nothing but a careless
choice of what to count — no third variable required, on data of exactly the kind you will
bring to this course yourself.